In [1]:
import pandas as pd
import numpy as np
import os

raw = "../data/raw/"
processed = "../data/processed/"
os.makedirs(processed, exist_ok=True)

# Load all needed tables
customers = pd.read_csv(raw + "olist_customers_dataset.csv")
orders    = pd.read_csv(raw + "olist_orders_dataset.csv")
items     = pd.read_csv(raw + "olist_order_items_dataset.csv")
products  = pd.read_csv(raw + "olist_products_dataset.csv")
payments  = pd.read_csv(raw + "olist_order_payments_dataset.csv")
reviews   = pd.read_csv(raw + "olist_order_reviews_dataset.csv")
sellers   = pd.read_csv(raw + "olist_sellers_dataset.csv")
geo       = pd.read_csv(raw + "olist_geolocation_dataset.csv")
trans     = pd.read_csv(raw + "product_category_name_translation.csv")

print("All tables loaded.")
for name, df in [("customers",customers),("orders",orders),("items",items),
                 ("products",products),("payments",payments),("reviews",reviews),
                 ("sellers",sellers),("geo",geo),("trans",trans)]:
    print(f"  {name:12s} {df.shape}")

All tables loaded.
  customers    (99441, 5)
  orders       (99441, 8)
  items        (112650, 7)
  products     (32951, 9)
  payments     (103886, 5)
  reviews      (99224, 7)
  sellers      (3095, 4)
  geo          (1000163, 5)
  trans        (71, 2)


In [2]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print(orders[date_cols].dtypes)
print("\nDate range:",
      orders["order_purchase_timestamp"].min(), "→",
      orders["order_purchase_timestamp"].max())

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Date range: 2016-09-04 21:15:19 → 2018-10-17 17:30:18


In [3]:
null_report = orders.isnull().sum().to_frame("nulls")
null_report["pct"] = (null_report["nulls"] / len(orders) * 100).round(2)
print(null_report[null_report["nulls"] > 0])

                               nulls   pct
order_approved_at                160  0.16
order_delivered_carrier_date    1783  1.79
order_delivered_customer_date   2965  2.98


In [4]:
impossible = orders[
    (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]) |
    (orders["order_approved_at"] < orders["order_purchase_timestamp"])
]
print(f"Impossible date rows: {len(impossible)}")
if len(impossible) > 0:
    print(impossible[["order_id","order_purchase_timestamp","order_approved_at",
                      "order_delivered_customer_date"]].head())

Impossible date rows: 0


In [5]:
print(orders["order_status"].value_counts())
print("\n% delivered:", round((orders["order_status"]=="delivered").mean()*100, 2))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

% delivered: 97.02


In [6]:
print("Missing product_category_name:", products["product_category_name"].isnull().sum())
products["product_category_name"] = products["product_category_name"].fillna("unknown")

Missing product_category_name: 610


In [7]:
products = products.merge(trans, on="product_category_name", how="left")
products["product_category_name_english"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])  # keep portuguese if no translation
)
print("Translation coverage:",
      round((products["product_category_name_english"] != products["product_category_name"]).mean()*100, 2), "%")

Translation coverage: 91.9 %


In [8]:
rename_map = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
}
products = products.rename(columns={k:v for k,v in rename_map.items() if k in products.columns})
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english']


In [9]:
products = products[[
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]]
print(products.shape)

(32951, 7)


In [10]:
print(items["price"].describe())
print("price <= 0:", (items["price"] <= 0).sum())
print("freight < 0:", (items["freight_value"] < 0).sum())

# Flag extreme prices (do NOT drop without business context)
q99 = items["price"].quantile(0.99)
print(f"99th percentile price: R$ {q99:.2f}")
print(f"Items above 99th percentile: {(items['price'] > q99).sum()}")

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64
price <= 0: 0
freight < 0: 0
99th percentile price: R$ 890.00
Items above 99th percentile: 1117


In [11]:
payments_agg = payments.groupby("order_id").agg(
    total_payment_value = ("payment_value", "sum"),
    n_payment_methods   = ("payment_type", "nunique"),
    max_installments    = ("payment_installments", "max"),
    primary_payment_type = ("payment_type", lambda s: s.mode().iloc[0] if len(s.mode()) else "unknown"),
).reset_index()
print(payments_agg.shape)
print(payments_agg.head())

(99440, 5)
                           order_id  total_payment_value  n_payment_methods  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19                  1   
1  00018f77f2f0320c557190d7a144bdd3               259.83                  1   
2  000229ec398224ef6ca0657da4fc703e               216.87                  1   
3  00024acbcdf0a6daa1e931b038114c75                25.78                  1   
4  00042b26cf59d7ce69dfabb4e55b4fd9               218.04                  1   

   max_installments primary_payment_type  
0                 2          credit_card  
1                 3          credit_card  
2                 5          credit_card  
3                 2          credit_card  
4                 3          credit_card  


In [12]:
print("reviews rows:", len(reviews))
print("unique review_id:", reviews["review_id"].nunique())
print("unique order_id:", reviews["order_id"].nunique())
print("missing review_score:", reviews["review_score"].isnull().sum())

# Keep one review per order — take the latest response
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")
reviews_clean = (reviews
    .sort_values("review_answer_timestamp")
    .drop_duplicates(subset="order_id", keep="last")
    [["order_id","review_score","review_comment_message"]]
)
print("reviews_clean rows:", len(reviews_clean))

reviews rows: 99224
unique review_id: 98410
unique order_id: 98673
missing review_score: 0
reviews_clean rows: 98673


In [13]:
print(customers.shape)
print("customer_id unique:", customers["customer_id"].is_unique)
print("unique customer_unique_id:", customers["customer_unique_id"].nunique())
print("customers rows:", len(customers))

(99441, 5)
customer_id unique: True
unique customer_unique_id: 96096
customers rows: 99441


In [14]:
# Step 1: Start with items (one row per order-item)
base = items.copy()

# Step 2: Filter to delivered orders only (revenue analysis)
orders_deliv = orders[orders["order_status"] == "delivered"].copy()

# Step 3: Join orders → keep purchase + delivery timestamps
base = base.merge(
    orders_deliv[[
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]],
    on="order_id", how="inner"
)

# Step 4: Join products (english category)
base = base.merge(
    products[["product_id", "product_category_name", "product_category_name_english"]],
    on="product_id", how="left"
)

# Step 5: Join customers (unique id + location)
base = base.merge(
    customers[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
    on="customer_id", how="left"
)

# Step 6: Join payments (aggregated)
base = base.merge(
    payments_agg[["order_id", "total_payment_value", "primary_payment_type", "max_installments"]],
    on="order_id", how="left"
)

# Step 7: Join reviews (one per order)
base = base.merge(
    reviews_clean[["order_id", "review_score"]],
    on="order_id", how="left"
)

print("order_master shape:", base.shape)
print(base.head(3))

order_master shape: (110197, 21)
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   

   shipping_limit_date  price  freight_value  \
0  2017-09-19 09:45:35   58.9          13.29   
1  2017-05-03 11:05:13  239.9          19.93   
2  2018-01-18 14:48:30  199.0          17.87   

                        customer_id order_status order_purchase_timestamp  \
0  3ce436f183e68e07877b285a838db11a    delivered      2017-09-13 08:59:02   
1  f6dd3ec061db4e3987629fe6b26e5cce    delivered      2017-04-26 10:53:06   
2  6489ae5e4333f3693df5ad4372dab6